# Деревья решений

In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
sys.path.append(str(Path.cwd().parent))
from evaluate_sku import (
    split_history,
    build_evaluation,
    compute_wape,
    compute_mase_per_sku,
    _get_month_columns
)

In [4]:
train = pd.read_csv('../data/train.csv')
print("Размер данных:", train.shape)
train.head()

Размер данных: (1656, 37)


,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,379.0,250.0,317.0,172.0,193.0,173.0,121.0,318.0,328.0,208.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,68.0,65.0,72.0,82.0,85.0,69.0,75.0,100.0,72.0,89.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,334.0,380.0,409.0,309.0,388.0,438.0,404.0,293.0,274.0,381.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0,2187.0,1672.0,1665.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,222.0,258.0,289.0,662.0,759.0,616.0,365.0,321.0,392.0,409.0


In [13]:
def extract_features(history: pd.DataFrame) -> pd.DataFrame:
    month_cols = _get_month_columns(history)
    features = pd.DataFrame(index=history.index)
    
    features['sku'] = history['sku']
    features['unit'] = history['unit']

    features['lag_1'] = history[month_cols[-1]]
    features['lag_2'] = history[month_cols[-2]]
    features['lag_3'] = history[month_cols[-3]]
    features['lag_12'] = history[month_cols[-12]]
    
    features['rolling_mean_3'] = history[month_cols[-3:]].mean(axis=1)
    features['rolling_mean_12'] = history[month_cols[-12:]].mean(axis=1)

    cutoff_month = int(month_cols[-1].split('-')[1])
    features['cutoff_month'] = cutoff_month
    
    return features

In [14]:
extract_features(train).head()

,sku,unit,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_12,cutoff_month
0,0,ШТ,208.0,328.0,318.0,271.0,284.666667,255.416667,11
1,1,ШТ,89.0,72.0,100.0,80.0,87.000000,75.333333,11
2,2,ШТ,381.0,274.0,293.0,327.0,316.000000,355.833333,11
3,3,ШТ,1665.0,1672.0,2187.0,1625.0,1841.333333,2128.583333,11
4,4,ШТ,409.0,392.0,321.0,599.0,374.000000,426.083333,11


Для кодирования колонки unit, мы выбрали OrdinalEncoder, потому что дереву не страшны нелинейные числа(они не умножают признак на веса) и он добавляет всего 1 столбец, а не 14 (для каждого типа в таблице)

In [7]:
unit_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
unit_encoder.fit(train[['unit']])

print("Уникальные категории:", len(unit_encoder.categories_[0]))

Уникальные категории: 14


In [16]:
def extract_features(history: pd.DataFrame, encoder: OrdinalEncoder) -> pd.DataFrame:
    month_cols = _get_month_columns(history)
    features = pd.DataFrame(index=history.index)
    
    features['sku'] = history['sku']
    features['unit_encoded'] = encoder.transform(history[['unit']]).astype(int) # добавил кодировку единиц измерений
    
    features['lag_1'] = history[month_cols[-1]]
    features['lag_2'] = history[month_cols[-2]]
    features['lag_3'] = history[month_cols[-3]]
    features['lag_12'] = history[month_cols[-12]]
    
    features['rolling_mean_3'] = history[month_cols[-3:]].mean(axis=1)
    features['rolling_mean_12'] = history[month_cols[-12:]].mean(axis=1)
    
    features['cutoff_month'] = int(month_cols[-1].split('-')[1])
    
    return features

In [17]:
features_df = extract_features(train, unit_encoder)
features_df.head()

,sku,unit_encoded,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_12,cutoff_month
0,0,13,208.0,328.0,318.0,271.0,284.666667,255.416667,11
1,1,13,89.0,72.0,100.0,80.0,87.000000,75.333333,11
2,2,13,381.0,274.0,293.0,327.0,316.000000,355.833333,11
3,3,13,1665.0,1672.0,2187.0,1625.0,1841.333333,2128.583333,11
4,4,13,409.0,392.0,321.0,599.0,374.000000,426.083333,11


В колонке unit появились числа, значит кодировка сработала. Теперь там 14 уникальных чисел

In [18]:
val_cutoff = '2021-08'
train_cutoff = '2021-05'

history_val, actuals_val = split_history(train, cutoff=val_cutoff)

history_train, actuals_train = split_history(train, cutoff=train_cutoff)

X_train_df = extract_features(history_train, unit_encoder)
X_val_df = extract_features(history_val, unit_encoder)

feature_cols = ['unit_encoded', 'lag_1', 'lag_2', 'lag_3', 'lag_12', 
                'rolling_mean_3', 'rolling_mean_12', 'cutoff_month']

X_train = X_train_df[feature_cols].values
X_val = X_val_df[feature_cols].values

target_train_months = ['2021-06', '2021-07', '2021-08']
y_train = train[target_train_months].values

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}")

X_train: (1656, 8), y_train: (1656, 3)
X_val:   (1656, 8)


Сформировали обучающую выборку со сдвигом на три месяца назад (отсечка 2021-05, таргет - лето 2021)

In [19]:
dt_model = DecisionTreeRegressor(random_state=1)
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=1)

In [22]:
y_val_pred = dt_model.predict(X_val)
print("Размерность предсказаний:", y_val_pred.shape)

Размерность предсказаний: (1656, 3)


Обучил дерево на многомерный таргет и получил матрица прогнозов

In [23]:
val_target_months = ['2021-09', '2021-10', '2021-11']

preds_wide = pd.DataFrame(y_val_pred, columns=val_target_months)
preds_wide['sku'] = history_val['sku']

preds_melted = preds_wide.melt(id_vars='sku', value_vars=val_target_months, 
                               var_name='month', value_name='prediction')
preds_melted['id'] = preds_melted['sku'].astype(str) + '_' + preds_melted['month']


eval_dt = build_evaluation(actuals_val, preds_melted[['id', 'prediction']])

wape_dt = compute_wape(eval_dt['actual'].to_numpy(), eval_dt['prediction'].to_numpy())
print(f"WAPE одиночного дерева решений (DecisionTree): {wape_dt:.4f}")

WAPE одиночного дерева решений (DecisionTree): 0.4916


In [25]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=1, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_val_pred_rf = rf_model.predict(X_val)

preds_wide_rf = pd.DataFrame(y_val_pred_rf, columns=val_target_months)
preds_wide_rf['sku'] = history_val['sku']

preds_melted_rf = preds_wide_rf.melt(id_vars='sku', value_vars=val_target_months, 
                                     var_name='month', value_name='prediction')
preds_melted_rf['id'] = preds_melted_rf['sku'].astype(str) + '_' + preds_melted_rf['month']

eval_rf = build_evaluation(actuals_val, preds_melted_rf[['id', 'prediction']])
wape_rf = compute_wape(eval_rf['actual'].to_numpy(), eval_rf['prediction'].to_numpy())
print(f"WAPE Случайного леса (RandomForest): {wape_rf:.4f}")

WAPE Случайного леса (RandomForest): 0.4191


Переход от одиночного дерева к ансаблю снизил показатель WAPE на 0,0725. Это произошло из-за того, что одно дерево склонно заучивать данные вместе с отклонениями, из-за чего сильно ошибается на новых данных. В случайном лесу каждое дерево учится на случайных частях данных. Мы получаем среднее значение этих деревьем, их промахи перерывают друг друга, поэтому результат становится точнее

### Подбор параметров

In [29]:
depths = [4, 6, 8, 10, 12, 15, None]
tuning_results = []

for depth in depths:
    rf_tune = RandomForestRegressor(
        n_estimators=100, 
        max_depth=depth, 
        min_samples_leaf=5, 
        random_state=1, 
        n_jobs=-1
    )
    rf_tune.fit(X_train, y_train)
    
    y_pred = rf_tune.predict(X_val)

    preds_wide = pd.DataFrame(y_pred, columns=val_target_months)
    preds_wide['sku'] = history_val['sku']
    preds_melted = preds_wide.melt(id_vars='sku', value_vars=val_target_months, 
                                   var_name='month', value_name='prediction')
    preds_melted['id'] = preds_melted['sku'].astype(str) + '_' + preds_melted['month']

    eval_df = build_evaluation(actuals_val, preds_melted[['id', 'prediction']])
    wape = compute_wape(eval_df['actual'].to_numpy(), eval_df['prediction'].to_numpy())
    
    tuning_results.append({
        'max_depth': str(depth),
        'WAPE': round(wape, 4)
    })

df_tuning = pd.DataFrame(tuning_results)
df_tuning

,max_depth,WAPE
0,4,0.4844
1,6,0.4612
2,8,0.4563
3,10,0.4556
4,12,0.4550
5,15,0.4514
6,None,0.4576


При малой глубе ошибка большая, так как деревья простые и  страдают от недообучения. При росте глубины до 15 показатель метрики улучшается.

In [30]:
leaf_samples = [1, 2, 3, 5]
leaf_results = []

for leaf in leaf_samples:
    rf_leaf = RandomForestRegressor(
        n_estimators=100, 
        max_depth=None, 
        min_samples_leaf=leaf, 
        random_state=1, 
        n_jobs=-1
    )
    rf_leaf.fit(X_train, y_train)
    y_pred = rf_leaf.predict(X_val)
    
    preds_wide = pd.DataFrame(y_pred, columns=val_target_months)
    preds_wide['sku'] = history_val['sku']
    preds_melted = preds_wide.melt(id_vars='sku', value_vars=val_target_months, 
                                   var_name='month', value_name='prediction')
    preds_melted['id'] = preds_melted['sku'].astype(str) + '_' + preds_melted['month']
    
    eval_df = build_evaluation(actuals_val, preds_melted[['id', 'prediction']])
    wape = compute_wape(eval_df['actual'].to_numpy(), eval_df['prediction'].to_numpy())
    
    leaf_results.append({
        'min_samples_leaf': leaf,
        'WAPE': round(wape, 4)
    })

pd.DataFrame(leaf_results)

,min_samples_leaf,WAPE
0,1,0.4191
1,2,0.4133
2,3,0.4376
3,5,0.4576


Значение 2 у параметра min_samples_leaf показало лучший результат среди всех моделей леса. Этот параметр отсекает единичные аномальные данные. При увеличении параметра, ошибка возрастает, потому что огрубление листов стирает индивидуальные паттерны

### Скользящаа валидация по 4 блокам

In [32]:
cutoffs = ['2020-11', '2021-02', '2021-05', '2021-08']
month_cols = _get_month_columns(train)

tree_results = []

for cutoff in cutoffs:
    # 1. Валидационные данные (наш экзамен)
    history_val, actuals_val = split_history(train, cutoff=cutoff)
    cutoff_idx = month_cols.index(cutoff)
    val_target_months = month_cols[cutoff_idx + 1 : cutoff_idx + 4]
    
    # 2. Обучающие данные (сдвигаемся на 3 месяца назад во времени)
    train_cutoff = month_cols[cutoff_idx - 3]
    history_train, actuals_train = split_history(train, cutoff=train_cutoff)
    train_cutoff_idx = month_cols.index(train_cutoff)
    target_train_months = month_cols[train_cutoff_idx + 1 : train_cutoff_idx + 4]
    
    # Считаем признаки для обучения и валидации
    X_train_df = extract_features(history_train, unit_encoder)
    X_val_df = extract_features(history_val, unit_encoder)
    
    X_train = X_train_df[feature_cols].values
    y_train = train[target_train_months].values
    X_val = X_val_df[feature_cols].values
    
    # 3. Обучаем наш лучший Случайный лес
    best_rf = RandomForestRegressor(
        n_estimators=100, 
        max_depth=None, 
        min_samples_leaf=2, 
        random_state=1, 
        n_jobs=-1
    )
    best_rf.fit(X_train, y_train)
    y_val_pred = best_rf.predict(X_val)
    
    # 4. Собираем предсказания в длинный формат
    preds_wide = pd.DataFrame(y_val_pred, columns=val_target_months)
    preds_wide['sku'] = history_val['sku']
    preds_melted = preds_wide.melt(id_vars='sku', value_vars=val_target_months, 
                                   var_name='month', value_name='prediction')
    preds_melted['id'] = preds_melted['sku'].astype(str) + '_' + preds_melted['month']
    
    # 5. Считаем WAPE
    eval_df = build_evaluation(actuals_val, preds_melted[['id', 'prediction']])
    wape = compute_wape(eval_df['actual'].to_numpy(), eval_df['prediction'].to_numpy())
    
    tree_results.append({
        'Период проверки': f"{val_target_months[0]} — {val_target_months[-1]}",
        'WAPE RandomForest': round(wape, 4)
    })

# Выводим таблицу
df_rf_summary = pd.DataFrame(tree_results)
df_rf_summary

,Период проверки,WAPE RandomForest
0,2020-12 — 2021-02,0.5410
1,2021-03 — 2021-05,0.4985
2,2021-06 — 2021-08,0.4152
3,2021-09 — 2021-11,0.4133


Провели скользящую валидацию, чтобы оценить модель в каждом сезоне
2020-12 — 2021-02 (Зима): отсечка 2020-11
2021-03 — 2021-05 (Весна): отсечка 2021-02
2021-06 — 2021-08 (Лето): отсечка 2021-05
2021-09 — 2021-11 (Осень): отсечка 2021-08
Для каждого окна модель обучается на истории с лагом в 3 месяца